# CUAD evaluation report (Colab)

One notebook that produces every number we might quote, on the **official CUAD test split (102 contracts)**, with nothing tuned on test.

**What it runs**

| Step | What | Uses test? |
|---|---|---|
| Run A | Exact reproduction of `train_and_evaluate` (all 408 train contracts, fixed 0.5 threshold) | evaluated once |
| Run B | Same trainer, but 15% of *train* contracts held out as validation; thresholds (global and per-label) tuned on that validation set only | evaluated once with the val-chosen thresholds |
| Baseline | TF-IDF (1-2 grams) + one logistic regression per category, same chunks, same masks, same max-pooling, same metric code | evaluated once |
| Risk layer | `src/risk` evidence + scorer + triage on top of each model, compared with classifier-only flags | evaluated once |
| Ranking | Recall of true high-risk items in the top 10/20/30% of a risk-sorted list, plus PR-AUC (average precision) | evaluated once |

**How to run:** Runtime -> Change runtime type -> **T4 GPU**, then Runtime -> Run all. You will be asked for a GitHub token (read access to the repo). Expect roughly an hour (two ~25 min training runs on a T4; the third optional run adds ~25 min). Set `USE_DRIVE = True` to cache predictions on Google Drive so a disconnect does not cost you a retrain.

At the end, `evaluation_results.zip` downloads with every table as CSV.

In [1]:
# ---- Settings -------------------------------------------------------------
REPO = "Break-Through-Tech/Accenture-1Q-contract-review-challenge"
BRANCH = "9-risk-scoring"          # the branch that has src/risk
USE_DRIVE = True                  # True: cache predictions in Drive across disconnects
RUN_VAL_MODEL = True               # Run B (needed for validation-tuned thresholds)
RUN_POS_WEIGHT_ABLATION = False    # extra ~25 min: class-imbalance (pos_weight) ablation
VAL_FRACTION = 0.15                # share of TRAIN contracts held out for threshold tuning
SPLIT_SEED = 42
MIN_VAL_SUPPORT_FOR_PER_LABEL = 3  # fewer val positives than this -> use the global threshold
N_BOOTSTRAP = 1000
TARGET_REVIEW_SHARE = 0.20         # risk layer: share of contracts the must-review rule should surface (fixed up front)

In [2]:
import getpass, os, subprocess, sys, time, json
from pathlib import Path

ROOT = Path("/content/repo")
if not ROOT.exists():
    token = getpass.getpass("GitHub token: ")
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH,
         f"https://{token}@github.com/{REPO}.git", str(ROOT)],
        check=True,
    )
    # Do not leave the token in .git/config.
    subprocess.run(["git", "-C", str(ROOT), "remote", "set-url", "origin", f"https://github.com/{REPO}.git"], check=True)
    del token
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = Path("/content/drive/MyDrive/cuad-eval")
else:
    OUT = ROOT / "reports/evaluation"
OUT.mkdir(parents=True, exist_ok=True)
print("Outputs ->", OUT)

642fb00 Add evidence extractor for risk scoring (#9)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Outputs -> /content/drive/MyDrive/cuad-eval


In [3]:
!pip install -q transformers pyyaml scikit-learn pyarrow

In [4]:
import numpy as np
import pandas as pd
import torch
import yaml
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from torch.utils.data import DataLoader
from transformers import AutoTokenizer

from src.eval.multilabel import multilabel_metrics
from src.ingest.chunker import get_tokenizer
from src.ingest.preprocessing import prepare_transformer_data
from src.models.trainer import _collate_chunks, train_and_evaluate
from src.models.transformer import prepare_label_arrays, select_train_positive_categories
from src.risk.evidence import extract_evidence
from src.risk.scorer import clause_signals, gold_predictions, load_rules, score_clauses
from src.risk.triage import triage_contracts

if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Runtime -> Change runtime type -> T4 GPU, then rerun.")
DEVICE = torch.device("cuda")
print(torch.cuda.get_device_name(0))
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

Tesla T4


## 1. Build chunks and labels (clean, then chunk)

`prepare_transformer_data` on this branch chunks the original text and cleans afterwards. The cleaner blanks artifacts with runs of spaces, which tokenize into extra tokens, so 4,397 of 17,079 `text_light` chunks exceed 510 tokens and the trainer refuses them. The first transformer run (`First_transformer.ipynb`, branch 43) avoided this by cleaning first. We do the same: clean each full contract, chunk the **cleaned** text, keep the original text at the same offsets for span labelling. The cleaner preserves length, so CUAD span offsets are unchanged. This reproduces that run's 17,324 chunks exactly.

In [5]:
from src.clean.light import clean_transformer_text
from src.ingest.chunker import chunk_contract
from src.ingest.label_chunks import label_chunks
from src.ingest.preprocessing import prepare_contract_categories

with (ROOT / "data/cuad/train_separate_questions.json").open(encoding="utf-8") as f:
    train_data = json.load(f)
with (ROOT / "data/cuad/test.json").open(encoding="utf-8") as f:
    test_data = json.load(f)

contract_cats, category_summary = prepare_contract_categories(train_data, test_data, cutoff=0.10)
print(category_summary)
tokenizer = get_tokenizer()
records = []
for split, dataset in (("train", train_data), ("test", test_data)):
    for doc in dataset["data"]:
        original = doc["paragraphs"][0]["context"]
        cleaned = clean_transformer_text(original)
        for ch in chunk_contract(cleaned, tokenizer=tokenizer):
            s, e = ch["start_char"], ch["end_char"]
            records.append({
                "split": split, "contract_id": doc["title"], "chunk_id": f"{doc['title']}__{ch['chunk_index']}",
                "chunk_index": ch["chunk_index"], "start_char": s, "end_char": e, "n_tokens": ch["n_tokens"],
                "text": original[s:e], "text_light": cleaned[s:e],
            })
chunks = pd.DataFrame(records)
chunk_labels = label_chunks(chunks, contract_cats, rule="any_overlap").chunk_labels
assert chunks["n_tokens"].max() <= 510

PROC = ROOT / "data/processed"
PROC.mkdir(parents=True, exist_ok=True)
chunks.to_parquet(PROC / "chunks.parquet", index=False)
chunk_labels.to_parquet(PROC / "chunk_labels.parquet", index=False)

META = {"split", "contract_id", "chunk_id", "chunk_index", "start_char", "end_char", "n_tokens"}
ALL_CATEGORIES = [c for c in chunk_labels.columns if c not in META]      # 41
CATEGORIES = select_train_positive_categories(chunk_labels)              # 34 model outputs
OUT_OF_SCOPE = [c for c in ALL_CATEGORIES if c not in CATEGORIES]
assert len(ALL_CATEGORIES) == 41 and len(CATEGORIES) == 34, (len(ALL_CATEGORIES), len(CATEGORIES))

TEST_IDS = list(chunk_labels.loc[chunk_labels["split"] == "test", "contract_id"].unique())
TRAIN_IDS = sorted(chunk_labels.loc[chunk_labels["split"] == "train", "contract_id"].unique())
print(f"chunks: {len(chunks):,}  train contracts: {len(TRAIN_IDS)}  test contracts: {len(TEST_IDS)}")
print("Out of model scope (no train positives after the 10% cutoff):", OUT_OF_SCOPE)

{'cutoff': 0.1, 'original_category_count': 41, 'excluded_categories': ['Affiliate License-Licensor', 'Most Favored Nation', 'Non-Disparagement', 'Price Restrictions', 'Source Code Escrow', 'Third Party Beneficiary', 'Unlimited/All-You-Can-Eat-License'], 'retained_below_cutoff': ['Joint Ip Ownership', 'No-Solicit Of Customers'], 'train_category_count': 34, 'test_category_count': 41}


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


chunks: 17,324  train contracts: 408  test contracts: 102
Out of model scope (no train positives after the 10% cutoff): ['Affiliate License-Licensor', 'Most Favored Nation', 'Non-Disparagement', 'Price Restrictions', 'Source Code Escrow', 'Third Party Beneficiary', 'Unlimited/All-You-Can-Eat-License']


### Leakage checks

In [6]:
import hashlib

def _h(text):
    return hashlib.sha1(" ".join(text.split()).encode("utf-8")).hexdigest()

contract_text = {
    split: {doc["title"]: _h(doc["paragraphs"][0]["context"]) for doc in data["data"]}
    for split, data in (("train", train_data), ("test", test_data))
}
title_overlap = set(contract_text["train"]) & set(contract_text["test"])
text_overlap = set(contract_text["train"].values()) & set(contract_text["test"].values())
train_chunk_hashes = set(chunks.loc[chunks["split"] == "train", "text"].map(_h))
test_chunk_dupes = chunks.loc[chunks["split"] == "test", "text"].map(_h).isin(train_chunk_hashes)

leakage = {
    "contracts in both splits (by title)": len(title_overlap),
    "test contracts whose full text also appears in train": len(text_overlap),
    "test chunks whose exact text also appears in train": f"{int(test_chunk_dupes.sum())} / {len(test_chunk_dupes)}",
}
for k, v in leakage.items():
    print(f"{k}: {v}")
assert not title_overlap, "a contract is in both splits"

contracts in both splits (by title): 0
test contracts whose full text also appears in train: 0
test chunks whose exact text also appears in train: 0 / 2938


## 2. Shared helpers

In [7]:
KEYS = ["split", "contract_id", "chunk_id"]
THRESHOLD_GRID = np.round(np.concatenate([[0.01, 0.02, 0.03, 0.04], np.arange(0.05, 0.951, 0.05)]), 2)


def contract_gold(contract_ids, categories):
    """Contract-level gold exactly as the trainer builds it (max over span-derived chunk labels)."""
    lab = chunk_labels[chunk_labels["contract_id"].isin(set(contract_ids))]
    gold = lab.groupby("contract_id", sort=False)[categories].max().reindex(contract_ids)
    return prepare_label_arrays(gold.reset_index(), categories)


def pool(chunk_probs, contract_ids):
    """Max-pool chunk probabilities to contract level (the trainer's aggregation)."""
    return chunk_probs.groupby("contract_id")[CATEGORIES].max().reindex(contract_ids).to_numpy(np.float32)


def wide_from_long(long):
    wide = long.pivot(index="chunk_id", columns="category", values="probability")[CATEGORIES]
    meta = long.drop_duplicates("chunk_id").set_index("chunk_id")[["contract_id"]]
    return meta.join(wide).reset_index()[["contract_id", "chunk_id"] + CATEGORIES]


def predict_chunk_probs(model, frame, batch_size=32):
    """Same tokenization/collation as the trainer's test loop."""
    tokenizer = AutoTokenizer.from_pretrained("distilroberta-base")
    dummy = np.zeros((len(frame), len(CATEGORIES)), np.float32)
    dataset = list(zip(frame["text_light"].tolist(), dummy, dummy.astype(bool)))
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False,
                        collate_fn=lambda b: _collate_chunks(b, tokenizer, 512))
    model.eval()
    out = []
    with torch.inference_mode():
        for batch in loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            batch.pop("labels"); batch.pop("label_mask")
            out.append(torch.sigmoid(model(**batch)["logits"]).float().cpu().numpy())
    result = frame[["contract_id", "chunk_id"]].reset_index(drop=True).copy()
    result[CATEGORIES] = np.concatenate(out)
    return result


def binarize(probs, thresholds):
    return (probs >= np.asarray(thresholds, dtype=np.float32)).astype(np.float32)


def score(targets, known, binary, categories):
    """Repo metric code; binary predictions scored at 0.5 so per-label thresholds work."""
    return multilabel_metrics(targets, binary, known, categories, threshold=0.5)


def tune_thresholds(val_targets, val_known, val_probs):
    """Global threshold maximising val macro-F1; per-label thresholds maximising val F1 per category."""
    global_scores = {t: score(val_targets, val_known, binarize(val_probs, t), CATEGORIES)[1]["macro_f1"]
                     for t in THRESHOLD_GRID}
    best_global = max(global_scores, key=lambda t: (global_scores[t], -abs(t - 0.5)))
    per_label = np.full(len(CATEGORIES), best_global, dtype=np.float32)
    for j in range(len(CATEGORIES)):
        valid = val_known[:, j]
        y = val_targets[valid, j].astype(bool)
        if y.sum() < MIN_VAL_SUPPORT_FOR_PER_LABEL:
            continue
        p = val_probs[valid, j]
        f1 = {}
        for t in THRESHOLD_GRID:
            pred = p >= t
            tp = (pred & y).sum(); fp = (pred & ~y).sum(); fn = (~pred & y).sum()
            f1[t] = 2 * tp / (2 * tp + fp + fn) if tp else 0.0
        per_label[j] = max(f1, key=lambda t: (f1[t], -abs(t - 0.5)))
    return float(best_global), per_label


def cached(name, fn):
    path = OUT / f"{name}.parquet"
    if path.exists():
        print(f"loaded cached {path.name}")
        return pd.read_parquet(path)
    frame = fn()
    frame.to_parquet(path, index=False)
    return frame


TEST_T, TEST_K = contract_gold(TEST_IDS, CATEGORIES)
TEST_T41, TEST_K41 = contract_gold(TEST_IDS, ALL_CATEGORIES)
print("test contract x category positives (34 labels):", int((TEST_T * TEST_K).sum()))

test contract x category positives (34 labels): 1218


## 3. Run A: the existing training run on the current pipeline

Uses `configs/transformer.yaml` and `train_and_evaluate` unchanged: all 408 train contracts, 3 epochs, threshold fixed at 0.5.

Same data construction as the first run (17,324 chunks), so this should land close to `reports/test_category_metrics.csv` (macro-F1 0.717 / micro-F1 0.851). GPU kernels are not bit-deterministic, so expect small differences; quote the numbers from this notebook.

In [8]:
def run_a():
    t0 = time.time()
    run = train_and_evaluate(ROOT / "configs/transformer.yaml", device="cuda")
    print(f"Run A took {(time.time() - t0) / 60:.1f} min")
    assert run["categories"] == CATEGORIES
    pd.DataFrame(run["history"]).to_csv(OUT / "runA_training_history.csv", index=False)
    wide = wide_from_long(run["test_chunk_predictions"])
    del run["model"]; torch.cuda.empty_cache()
    return wide

A_test = cached("runA_test_chunk_probs", run_a)
P_A = pool(A_test, TEST_IDS)
print("Run A @0.5:", score(TEST_T, TEST_K, binarize(P_A, 0.5), CATEGORIES)[1])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: distilroberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


epoch 1/3 - train loss: 0.0887
epoch 2/3 - train loss: 0.0470
epoch 3/3 - train loss: 0.0354


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

test summary: {
  "threshold": 0.5,
  "n_categories": 34,
  "macro_precision": 0.82126100733403,
  "macro_recall": 0.6954413123269773,
  "macro_f1": 0.716935402962176,
  "micro_precision": 0.8700516351118761,
  "micro_recall": 0.8300492610837439,
  "micro_f1": 0.8495798319327732
}
Run A took 40.7 min
Run A @0.5: {'threshold': 0.5, 'n_categories': 34, 'macro_precision': 0.82126100733403, 'macro_recall': 0.6954413123269773, 'macro_f1': 0.716935402962176, 'micro_precision': 0.8700516351118761, 'micro_recall': 0.8300492610837439, 'micro_f1': 0.8495798319327732}


## 4. Run B: hold out validation contracts from train, tune thresholds there

Validation contracts come only from the official train split. The split seed is advanced until every one of the 34 categories still has a positive in both the reduced train set (the trainer requires this) and the validation set.

In [9]:
def choose_val_ids():
    n_val = round(VAL_FRACTION * len(TRAIN_IDS))
    train_rows = chunk_labels["split"] == "train"
    fallback = None
    for seed in range(SPLIT_SEED, SPLIT_SEED + 500):
        val = set(np.random.default_rng(seed).choice(TRAIN_IDS, size=n_val, replace=False))
        in_val = chunk_labels["contract_id"].isin(val)
        tr_ok = chunk_labels.loc[train_rows & ~in_val, CATEGORIES].fillna(False).astype(bool).any().all()
        va_ok = chunk_labels.loc[in_val, CATEGORIES].fillna(False).astype(bool).any().all()
        if tr_ok and va_ok:
            return sorted(val), seed
        if tr_ok and fallback is None:
            fallback = (sorted(val), seed)
    return fallback

VAL_IDS, used_seed = choose_val_ids()
SUB_TRAIN_IDS = sorted(set(TRAIN_IDS) - set(VAL_IDS))
assert not set(VAL_IDS) & set(TEST_IDS)
print(f"val contracts: {len(VAL_IDS)} (seed {used_seed}); reduced train: {len(SUB_TRAIN_IDS)}")
pd.Series(VAL_IDS, name="contract_id").to_csv(OUT / "val_contract_ids.csv", index=False)

VAL_T, VAL_K = contract_gold(VAL_IDS, CATEGORIES)
val_chunks = chunks[chunks["contract_id"].isin(set(VAL_IDS))].reset_index(drop=True)


def write_config(name, **overrides):
    with (ROOT / "configs/transformer.yaml").open() as f:
        cfg = yaml.safe_load(f)
    for dotted, value in overrides.items():
        section, key = dotted.split(".")
        cfg[section][key] = value
    path = ROOT / f"configs/{name}.yaml"
    path.write_text(yaml.safe_dump(cfg, sort_keys=False))
    return path


def run_b():
    in_val = chunks["contract_id"].isin(set(VAL_IDS))
    chunks_b, labels_b = chunks.copy(), chunk_labels.copy()
    chunks_b.loc[in_val, "split"] = "val"           # trainer only reads split == train / test
    labels_b.loc[labels_b["contract_id"].isin(set(VAL_IDS)), "split"] = "val"
    chunks_b.to_parquet(PROC / "chunks_valsplit.parquet", index=False)
    labels_b.to_parquet(PROC / "chunk_labels_valsplit.parquet", index=False)
    cfg = write_config(
        "transformer_valsplit",
        **{"data.chunks_path": "data/processed/chunks_valsplit.parquet",
           "data.labels_path": "data/processed/chunk_labels_valsplit.parquet",
           "artifacts.output_dir": "models/distilroberta-cuad-valsplit"},
    )
    t0 = time.time()
    run = train_and_evaluate(cfg, device="cuda")
    print(f"Run B took {(time.time() - t0) / 60:.1f} min")
    assert run["categories"] == CATEGORIES
    pd.DataFrame(run["history"]).to_csv(OUT / "runB_training_history.csv", index=False)
    test_wide = wide_from_long(run["test_chunk_predictions"])
    val_wide = predict_chunk_probs(run["model"], val_chunks)
    del run["model"]; torch.cuda.empty_cache()
    val_wide.to_parquet(OUT / "runB_val_chunk_probs.parquet", index=False)
    return test_wide

if RUN_VAL_MODEL:
    B_test = cached("runB_test_chunk_probs", run_b)
    B_val = pd.read_parquet(OUT / "runB_val_chunk_probs.parquet")
    P_B, P_B_val = pool(B_test, TEST_IDS), pool(B_val, VAL_IDS)
    B_global, B_per_label = tune_thresholds(VAL_T, VAL_K, P_B_val)   # VALIDATION ONLY
    print("Run B val-tuned global threshold:", B_global)
    print("Run B val-tuned per-label thresholds:", dict(zip(CATEGORIES, B_per_label.round(2))))

val contracts: 61 (seed 42); reduced train: 347


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: distilroberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


epoch 1/3 - train loss: 0.0915
epoch 2/3 - train loss: 0.0482
epoch 3/3 - train loss: 0.0368


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

test summary: {
  "threshold": 0.5,
  "n_categories": 34,
  "macro_precision": 0.7474680457614922,
  "macro_recall": 0.6264302041114341,
  "macro_f1": 0.6565995524310932,
  "micro_precision": 0.8895131086142322,
  "micro_recall": 0.7799671592775042,
  "micro_f1": 0.8311461067366579
}
Run B took 34.9 min
Run B val-tuned global threshold: 0.15
Run B val-tuned per-label thresholds: {'Affiliate License-Licensee': np.float32(0.15), 'Agreement Date': np.float32(0.35), 'Anti-Assignment': np.float32(0.1), 'Audit Rights': np.float32(0.5), 'Cap On Liability': np.float32(0.3), 'Change Of Control': np.float32(0.3), 'Competitive Restriction Exception': np.float32(0.1), 'Covenant Not To Sue': np.float32(0.65), 'Document Name': np.float32(0.05), 'Effective Date': np.float32(0.35), 'Exclusivity': np.float32(0.35), 'Expiration Date': np.float32(0.5), 'Governing Law': np.float32(0.5), 'Insurance': np.float32(0.5), 'Ip Ownership Assignment': np.float32(0.15), 'Irrevocable Or Perpetual License': np.float3

## 5. Optional ablation: class-imbalance weighting (`imbalance_strategy: pos_weight`)

Implemented in the trainer but never used (`none` in the config). Same data as Run A, threshold 0.5. This is an ablation, not a model-selection step: picking whichever variant scores best on test would be selection on test.

In [10]:
def run_pw():
    cfg = write_config("transformer_posweight",
                       **{"training.imbalance_strategy": "pos_weight",
                          "artifacts.output_dir": "models/distilroberta-cuad-posweight"})
    run = train_and_evaluate(cfg, device="cuda")
    wide = wide_from_long(run["test_chunk_predictions"])
    del run["model"]; torch.cuda.empty_cache()
    return wide

if RUN_POS_WEIGHT_ABLATION:
    PW_test = cached("runPW_test_chunk_probs", run_pw)
    P_PW = pool(PW_test, TEST_IDS)

## 6. Baseline: TF-IDF + one-vs-rest logistic regression

Trained on the same chunks (`text_light`), masked the same way (unknown labels dropped per category), max-pooled to contracts, scored with the same code. Hyperparameters are fixed up front (no tuning on test). Two fits: all train (compare with Run A at 0.5) and reduced train (thresholds tuned on the same validation contracts as Run B).

In [11]:
data = chunks[KEYS + ["text_light"]].merge(chunk_labels[KEYS + ALL_CATEGORIES], on=KEYS, validate="one_to_one")


def fit_tfidf_lr(train_ids, predict_frames):
    train_frame = data[data["contract_id"].isin(set(train_ids))]
    vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.9, sublinear_tf=True,
                          max_features=300_000, dtype=np.float32)
    X_train = vec.fit_transform(train_frame["text_light"])
    Y, K = prepare_label_arrays(train_frame, CATEGORIES)
    X_pred = [vec.transform(f["text_light"]) for f in predict_frames]
    outs = [f[["contract_id", "chunk_id"]].reset_index(drop=True).copy() for f in predict_frames]
    probs = [np.zeros((len(f), len(CATEGORIES)), np.float32) for f in predict_frames]
    for j in range(len(CATEGORIES)):
        rows = K[:, j]
        clf = LogisticRegression(C=4.0, solver="liblinear", max_iter=2000)
        clf.fit(X_train[rows], Y[rows, j])
        for p, X in zip(probs, X_pred):
            p[:, j] = clf.predict_proba(X)[:, 1]
    for o, p in zip(outs, probs):
        o[CATEGORIES] = p
    return outs

test_frame = data[data["split"] == "test"]
val_frame = data[data["contract_id"].isin(set(VAL_IDS))]
t0 = time.time()
(BL_test,) = fit_tfidf_lr(TRAIN_IDS, [test_frame])
BLsub_test, BLsub_val = fit_tfidf_lr(SUB_TRAIN_IDS, [test_frame, val_frame])
print(f"baseline fits took {(time.time() - t0) / 60:.1f} min")
P_BL, P_BLsub, P_BLsub_val = pool(BL_test, TEST_IDS), pool(BLsub_test, TEST_IDS), pool(BLsub_val, VAL_IDS)
BL_global, BL_per_label = tune_thresholds(VAL_T, VAL_K, P_BLsub_val)   # VALIDATION ONLY
print("Baseline val-tuned global threshold:", BL_global)

baseline fits took 2.4 min
Baseline val-tuned global threshold: 0.1


## 7. Risk layer set-up and ground truth

There is no human "high-risk" label in CUAD, so two targets are used, both built from CUAD's human clause annotations:

* **T1, high-risk category present:** the contract contains a clause from one of the categories `configs/risk_rules.yaml` tiers as `high`.
* **T2, rule-engine High on gold labels:** the risk engine, fed CUAD's human labels and the annotated clause text (`gold_predictions`), rates the clause High. This measures how faithfully the predicted pipeline reproduces the rule engine's verdict on human labels. It is not an independent legal judgment.

Framings compared on each target:

* **Classifier only:** flag = classifier predicts a high-tier category.
* **With risk layer:** flag = `score_clauses` rates the detected clause High (evidence = best chunk, as in `extract_evidence`).

### Re-pick the High cutoff on validation contracts

`risk_rules.yaml` set High = 0.70 by looking at how many **test** contracts each cutoff sent to must-review (0.60 -> 71 of 102, 0.70 -> 20). Here the same criterion is applied to the **validation** contracts only: pick the cutoff whose must-review share on gold labels is closest to `TARGET_REVIEW_SHARE` (ties go to the cutoff nearest the original 0.70). Gold clause scores are discrete, so many cutoffs are equivalent. Every risk-layer number below uses the validation-picked cutoff.

In [12]:
import copy

rules_original = load_rules()
must = rules_original["triage"]["always_review_if_high_count"]
val_cats = contract_cats[contract_cats["contract_id"].isin(set(VAL_IDS)) & contract_cats["category"].isin(CATEGORIES)]
val_gold_clauses = gold_predictions(val_cats)
choices = []
for cutoff in np.round(np.arange(0.55, 1.001, 0.05), 2):
    r = copy.deepcopy(rules_original)
    r["thresholds"]["high"] = float(cutoff)
    reg = score_clauses(val_gold_clauses, r, run_id="val-gold")
    tri = triage_contracts(reg, r, contract_ids=VAL_IDS)
    share = (tri["n_high"] >= must).mean()
    choices.append({"high_cutoff": float(cutoff), "val_must_review_share": share,
                    "val_contracts_flagged": int((tri["n_high"] >= must).sum())})
choices = pd.DataFrame(choices)
choices["distance"] = (choices["val_must_review_share"] - TARGET_REVIEW_SHARE).abs()
choices["from_original"] = (choices["high_cutoff"] - rules_original["thresholds"]["high"]).abs()
display(choices)
picked = float(choices.sort_values(["distance", "from_original"]).iloc[0]["high_cutoff"])
choices.to_csv(OUT / "risk_cutoff_selection_on_val.csv", index=False)

rules = copy.deepcopy(rules_original)
rules["thresholds"]["high"] = picked
print(f"High cutoff: original {rules_original['thresholds']['high']} (chosen on test) -> {picked} (chosen on validation)")

,high_cutoff,val_must_review_share,val_contracts_flagged,distance,from_original
0,0.55,0.655738,40,0.455738,0.15
1,0.60,0.655738,40,0.455738,0.10
2,0.65,0.180328,11,0.019672,0.05
3,0.70,0.180328,11,0.019672,0.00
4,0.75,0.180328,11,0.019672,0.05
5,0.80,0.180328,11,0.019672,0.10
6,0.85,0.032787,2,0.167213,0.15
7,0.90,0.032787,2,0.167213,0.20
8,0.95,0.032787,2,0.167213,0.25
9,1.00,0.032787,2,0.167213,0.30


High cutoff: original 0.7 (chosen on test) -> 0.7 (chosen on validation)


In [13]:
tiers = rules["category_tiers"]
HIGH = [c for c in CATEGORIES if tiers.get(c, tiers["default"]) == "high"]
HIGH_IDX = [CATEGORIES.index(c) for c in HIGH]
print("high-tier categories:", HIGH)

test_cats = contract_cats[(contract_cats["split"] == "test") & contract_cats["category"].isin(CATEGORIES)]
gold_register = score_clauses(gold_predictions(test_cats), rules, run_id="gold")
gold_high = set(map(tuple, gold_register.loc[gold_register["risk_level"] == "High", ["contract_id", "category"]].to_numpy()))
cid_index = {c: i for i, c in enumerate(TEST_IDS)}
cat_index = {c: j for j, c in enumerate(CATEGORIES)}

Y_T2 = np.zeros_like(TEST_T, dtype=bool)
for cid, cat in gold_high:
    Y_T2[cid_index[cid], cat_index[cat]] = True
Y_T1 = TEST_T.astype(bool)                     # restricted to HIGH columns where used
U_T1 = np.zeros_like(TEST_K); U_T1[:, HIGH_IDX] = TEST_K[:, HIGH_IDX]   # universe for T1
U_T2 = TEST_K.copy()                                                    # universe for T2
Y_T2 &= U_T2

gold_triage = triage_contracts(gold_register, rules, contract_ids=TEST_IDS).set_index("contract_id").reindex(TEST_IDS)
Y_CONTRACT = (gold_triage["n_high"] >= must).to_numpy()
Y_CONTRACT_ANY = (gold_triage["n_high"] >= 1).to_numpy()
print(f"T1 positives: {int((Y_T1 & U_T1.astype(bool)).sum())} of {int(U_T1.sum())} pairs")
print(f"T2 positives: {int(Y_T2.sum())} of {int(U_T2.sum())} pairs")
print(f"gold high-risk contracts (>= {must} High clauses): {int(Y_CONTRACT.sum())} of {len(TEST_IDS)}; (>= 1): {int(Y_CONTRACT_ANY.sum())}")

high-tier categories: ['Change Of Control', 'Exclusivity', 'Ip Ownership Assignment', 'Liquidated Damages', 'Minimum Commitment', 'Non-Compete', 'Uncapped Liability']
T1 positives: 164 of 714 pairs
T2 positives: 77 of 3468 pairs
gold high-risk contracts (>= 2 High clauses): 20 of 102; (>= 1): 49


In [14]:
weights = {name: s["weight"] for name, s in rules["signals"].items()}


def prf(y, flag, universe):
    u = universe.astype(bool)
    y, flag = y[u], flag[u]
    tp = (y & flag).sum(); fp = (~y & flag).sum(); fn = (y & ~flag).sum()
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    return {"precision": p, "recall": r, "f1": 2 * p * r / (p + r) if p + r else 0.0}


def recall_at(scores, y, fracs=(0.1, 0.2, 0.3)):
    order = np.argsort(-scores, kind="stable")
    n, pos = len(y), y.sum()
    out = {}
    for f in fracs:
        k = int(np.ceil(f * n))
        out[f"recall@top{int(f * 100)}%"] = y[order[:k]].sum() / pos
        out[f"ceiling@top{int(f * 100)}%"] = min(1.0, k / pos)
    out["pr_auc"] = average_precision_score(y, scores)
    out["pr_auc_random"] = pos / n
    out["n_items"], out["n_pos"] = n, int(pos)
    return out


def risk_outputs(chunk_probs, P, thresholds, label):
    """Run the risk layer on one model's test predictions."""
    thr = np.broadcast_to(np.asarray(thresholds, dtype=np.float32), (len(CATEGORIES),))
    thr_map = dict(zip(CATEGORIES, thr))
    long = chunk_probs.melt(id_vars=["contract_id", "chunk_id"], value_vars=CATEGORIES,
                            var_name="category", value_name="probability")
    long["predicted_label"] = long["probability"] >= long["category"].map(thr_map)
    register = score_clauses(extract_evidence(long, chunks), rules, run_id=label)

    risk_flag = np.zeros_like(TEST_T, dtype=bool)
    for cid, cat in register.loc[register["risk_level"] == "High", ["contract_id", "category"]].to_numpy():
        risk_flag[cid_index[cid], cat_index[cat]] = True

    # Expected risk for every pair = P(clause present) x rule score if it were present.
    detected = binarize(P, thr).astype(bool)
    detected_sets = {cid: {CATEGORIES[j] for j in np.flatnonzero(detected[i])} for i, cid in enumerate(TEST_IDS)}
    best = (long.sort_values("probability", ascending=False)
            .drop_duplicates(["contract_id", "category"])
            .merge(chunks[["chunk_id", "text"]], on="chunk_id"))
    expected = np.zeros_like(P)
    for row in best.itertuples(index=False):
        sig = clause_signals(row.category, row.probability, row.text, detected_sets[row.contract_id], rules)
        expected[cid_index[row.contract_id], cat_index[row.category]] = row.probability * sum(weights[k] * v for k, v in sig.items())

    triage = triage_contracts(register, rules, contract_ids=TEST_IDS).set_index("contract_id").reindex(TEST_IDS)
    return {"detected": detected, "risk_flag": risk_flag, "expected": expected, "triage": triage}

## 8. Evaluate every configuration

In [15]:
configs = [
    ("TF-IDF + LR (all train)", "0.5 fixed", BL_test, P_BL, 0.5),
    ("TF-IDF + LR (85% train)", "val-tuned global", BLsub_test, P_BLsub, BL_global),
    ("TF-IDF + LR (85% train)", "val-tuned per-label", BLsub_test, P_BLsub, BL_per_label),
    ("DistilRoBERTa run A (all train)", "0.5 fixed", A_test, P_A, 0.5),
]
if RUN_VAL_MODEL:
    configs += [
        ("DistilRoBERTa run B (85% train)", "0.5 fixed", B_test, P_B, 0.5),
        ("DistilRoBERTa run B (85% train)", "val-tuned global", B_test, P_B, B_global),
        ("DistilRoBERTa run B (85% train)", "val-tuned per-label", B_test, P_B, B_per_label),
    ]
if RUN_POS_WEIGHT_ABLATION:
    configs.append(("DistilRoBERTa pos_weight (ablation)", "0.5 fixed", PW_test, P_PW, 0.5))

# Chunking ablation: score only each contract's first 510-token window (no retraining).
first_chunk = A_test[A_test["chunk_id"].str.rsplit("__", n=1).str[1].astype(int) == 0]
P_A_first = pool(first_chunk, TEST_IDS)

rows, per_category_tables, risk_cache = [], {}, {}


def add(model, thr_name, section, scope, framing, metrics):
    for k, v in metrics.items():
        rows.append({"model": model, "threshold": thr_name, "section": section,
                     "scope": scope, "framing": framing, "metric": k, "value": float(v)})


def classifier_metrics(model, thr_name, P, thresholds):
    B = binarize(P, thresholds)
    per, s34 = score(TEST_T, TEST_K, B, CATEGORIES)
    per_category_tables[(model, thr_name)] = per
    keep = ["micro_precision", "micro_recall", "micro_f1", "macro_precision", "macro_recall", "macro_f1"]
    add(model, thr_name, "classifier", "34 modeled labels", "classifier", {k: s34[k] for k in keep})
    _, s_hi = score(TEST_T[:, HIGH_IDX], TEST_K[:, HIGH_IDX], B[:, HIGH_IDX], HIGH)
    add(model, thr_name, "classifier", f"{len(HIGH)} high-risk categories", "classifier", {k: s_hi[k] for k in keep})
    # 41-label view: the 7 out-of-scope categories are always predicted absent.
    B41 = np.zeros_like(TEST_T41)
    for j, c in enumerate(ALL_CATEGORIES):
        if c in cat_index:
            B41[:, j] = B[:, cat_index[c]]
    per41, s41 = score(TEST_T41, TEST_K41, B41, ALL_CATEGORIES)
    supported = per41["support"] > 0
    s41["macro_f1_labels_with_test_support"] = per41.loc[supported, "f1"].mean()
    add(model, thr_name, "classifier", "41 CUAD labels (7 never predicted)", "classifier",
        {k: s41[k] for k in keep + ["macro_f1_labels_with_test_support"]})
    return B


for model, thr_name, chunk_probs, P, thresholds in configs:
    B = classifier_metrics(model, thr_name, P, thresholds)
    r = risk_outputs(chunk_probs, P, thresholds, label=f"{model} | {thr_name}")
    risk_cache[(model, thr_name)] = r
    cls_flag = np.zeros_like(B, dtype=bool); cls_flag[:, HIGH_IDX] = B[:, HIGH_IDX].astype(bool)

    for target, y, universe in (("T1 high-tier category present", Y_T1, U_T1), ("T2 rule-engine High on gold", Y_T2, U_T2)):
        add(model, thr_name, "high-risk detection", target, "classifier only", prf(y, cls_flag, universe))
        add(model, thr_name, "high-risk detection", target, "with risk layer", prf(y, r["risk_flag"], universe))

        u = universe.astype(bool)
        cls_score = np.where(np.isin(np.arange(len(CATEGORIES)), HIGH_IDX)[None, :], P, 0.0)
        add(model, thr_name, "clause ranking", target, "classifier only (P x high-tier)", recall_at(cls_score[u], y[u]))
        add(model, thr_name, "clause ranking", target, "with risk layer (P x rule score)", recall_at(r["expected"][u], y[u]))

    contract_cls = P[:, HIGH_IDX].sum(axis=1)                      # expected # of high-tier clause types
    contract_risk = -r["triage"]["review_rank"].to_numpy(float)    # the triage order itself
    for target, y in ((f"contracts with >= {must} gold High clauses", Y_CONTRACT), ("contracts with >= 1 gold High clause", Y_CONTRACT_ANY)):
        add(model, thr_name, "contract ranking", target, "classifier only (sum P high-tier)", recall_at(contract_cls, y))
        add(model, thr_name, "contract ranking", target, "with risk layer (triage rank)", recall_at(contract_risk, y))

classifier_metrics("DistilRoBERTa run A, first window only (chunking ablation)", "0.5 fixed", P_A_first, 0.5)
results = pd.DataFrame(rows)
results.to_csv(OUT / "all_metrics_long.csv", index=False)
print(len(results), "metric rows")

796 metric rows


## 9. Tables

In [16]:
def view(section, metrics, framings=None):
    d = results[(results["section"] == section) & results["metric"].isin(metrics)]
    if framings:
        d = d[d["framing"].isin(framings)]
    t = d.pivot_table(index=["model", "threshold"], columns=["scope", "framing", "metric"], values="value", sort=False)
    return t.round(3)

print("CLASSIFIER (contract-level presence, official test, 102 contracts)")
cls_table = view("classifier", ["micro_precision", "micro_recall", "micro_f1", "macro_precision", "macro_recall", "macro_f1",
                                "macro_f1_labels_with_test_support"])
display(cls_table)
cls_table.to_csv(OUT / "table_classifier.csv")

CLASSIFIER (contract-level presence, official test, 102 contracts)


scope                                                                  34 modeled labels                                                             7 high-risk categories                        \
framing                                                                       classifier                                                                         classifier                         
metric                                                                   micro_precision micro_recall micro_f1 macro_precision macro_recall macro_f1        micro_precision micro_recall micro_f1   
model                                              threshold                                                                                                                                        
TF-IDF + LR (all train)                            0.5 fixed                       0.952        0.532    0.682           0.830        0.389    0.495                  0.921        0.213    0.347   
TF-IDF + LR (85% train)                            val-tuned global                0.820        0.847    0.834           0.781        0.740    0.729                  0.704        0.726    0.715   
                                                   val-tuned per-label             0.797        0.860    0.827           0.746        0.776    0.727                  0.646        0.701    0.673   
DistilRoBERTa run A (all train)                    0.5 fixed                       0.870        0.830    0.850           0.821        0.695    0.717                  0.784        0.665    0.719   
DistilRoBERTa run B (85% train)                    0.5 fixed                       0.890        0.780    0.831           0.747        0.626    0.657                  0.850        0.555    0.672   
                                                   val-tuned global                0.736        0.918    0.817           0.639        0.835    0.708                  0.591        0.854    0.698   
                                                   val-tuned per-label             0.797        0.868    0.831           0.737        0.788    0.736                  0.625        0.762    0.687   
DistilRoBERTa run A, first window only (chunkin... 0.5 fixed                       0.919        0.280    0.429           0.302        0.119    0.138                  0.667        0.012    0.024   

scope                                                                                                        41 CUAD labels (7 never predicted)                                                     \
framing                                                                                                                              classifier                                                      
metric                                                                 macro_precision macro_recall macro_f1                    micro_precision micro_recall micro_f1 macro_precision macro_recall   
model                                              threshold                                                                                                                                         
TF-IDF + LR (all train)                            0.5 fixed                     0.812        0.224    0.324                              0.952        0.521    0.673           0.688        0.322   
TF-IDF + LR (85% train)                            val-tuned global              0.740        0.716    0.692                              0.820        0.830    0.825           0.648        0.613   
                                                   val-tuned per-label           0.721        0.678    0.607                              0.797        0.842    0.819           0.619        0.643   
DistilRoBERTa run A (all train)                    0.5 fixed                     0.802        0.639    0.669                              0.870        0.813    0.840           0.681        0.577   
DistilRoBERTa run B (85% train)                    

In [17]:
print("HIGH-RISK DETECTION, with vs without the risk layer (contract x category pairs)")
det_table = view("high-risk detection", ["precision", "recall", "f1"])
display(det_table)
det_table.to_csv(OUT / "table_high_risk_detection.csv")

HIGH-RISK DETECTION, with vs without the risk layer (contract x category pairs)


scope                                               T1 high-tier category present                                             T2 rule-engine High on gold                                            
framing                                                           classifier only               with risk layer                           classifier only               with risk layer              
metric                                                                  precision recall     f1       precision recall     f1                   precision recall     f1       precision recall     f1
model                           threshold                                                                                                                                                            
TF-IDF + LR (all train)         0.5 fixed                                   0.921  0.213  0.347           0.897  0.159  0.269                       0.579  0.286  0.383           0.759  0.286  0.415
TF-IDF + LR (85% train)         val-tuned global                            0.704  0.726  0.715           0.796  0.238  0.366                       0.343  0.753  0.472           0.673  0.429  0.524
                                val-tuned per-label                         0.646  0.701  0.673           0.600  0.146  0.235                       0.326  0.753  0.455           0.550  0.286  0.376
DistilRoBERTa run A (all train) 0.5 fixed                                   0.784  0.665  0.719           0.823  0.396  0.535                       0.396  0.714  0.509           0.612  0.636  0.624
DistilRoBERTa run B (85% train) 0.5 fixed                                   0.850  0.555  0.672           0.877  0.390  0.540                       0.467  0.649  0.543           0.644  0.610  0.627
                                val-tuned global                            0.591  0.854  0.698           0.831  0.329  0.472                       0.283  0.870  0.427           0.677  0.571  0.620
                                val-tuned per-label                         0.625  0.762  0.687           0.869  0.323  0.471                       0.315  0.818  0.455           0.705  0.558  0.623

In [18]:
rank_metrics = ["recall@top10%", "recall@top20%", "recall@top30%", "ceiling@top10%", "ceiling@top20%", "ceiling@top30%",
                "pr_auc", "pr_auc_random", "n_items", "n_pos"]
print("CLAUSE RANKING (pairs sorted by predicted risk)")
clause_rank = view("clause ranking", rank_metrics)
display(clause_rank)
clause_rank.to_csv(OUT / "table_clause_ranking.csv")
print("CONTRACT RANKING (contracts sorted by predicted risk)")
contract_rank = view("contract ranking", rank_metrics)
display(contract_rank)
contract_rank.to_csv(OUT / "table_contract_ranking.csv")

CLAUSE RANKING (pairs sorted by predicted risk)


scope                                                 T1 high-tier category present                                                                                                               \
framing                                             classifier only (P x high-tier)                                                                                                                
metric                                                                recall@top10% ceiling@top10% recall@top20% ceiling@top20% recall@top30% ceiling@top30% pr_auc pr_auc_random n_items  n_pos   
model                           threshold                                                                                                                                                          
TF-IDF + LR (all train)         0.5 fixed                                     0.402          0.439         0.671          0.872         0.829            1.0  0.801          0.23   714.0  164.0   
TF-IDF + LR (85% train)         val-tuned global                              0.390          0.439         0.646          0.872         0.811            1.0  0.785          0.23   714.0  164.0   
                                val-tuned per-label                           0.390          0.439         0.646          0.872         0.811            1.0  0.785          0.23   714.0  164.0   
DistilRoBERTa run A (all train) 0.5 fixed                                     0.415          0.439         0.677          0.872         0.817            1.0  0.833          0.23   714.0  164.0   
DistilRoBERTa run B (85% train) 0.5 fixed                                     0.409          0.439         0.677          0.872         0.823            1.0  0.815          0.23   714.0  164.0   
                                val-tuned global                              0.409          0.439         0.677          0.872         0.823            1.0  0.815          0.23   714.0  164.0   
                                val-tuned per-label                           0.409          0.439         0.677          0.872         0.823            1.0  0.815          0.23   714.0  164.0   

scope                                                                                                                                                                                              \
framing                                             with risk layer (P x rule score)                                                                                                                
metric                                                                 recall@top10% ceiling@top10% recall@top20% ceiling@top20% recall@top30% ceiling@top30% pr_auc pr_auc_random n_items  n_pos   
model                           threshold                                                                                                                                                           
TF-IDF + LR (all train)         0.5 fixed                                      0.415          0.439         0.652          0.872         0.829            1.0  0.796          0.23   714.0  164.0   
TF-IDF + LR (85% train)         val-tuned global                               0.402          0.439         0.640          0.872         0.829            1.0  0.784          0.23   714.0  164.0   
                                val-tuned per-label                            0.390          0.439         0.634          0.872         0.817            1.0  0.774          0.23   714.0  164.0   
DistilRoBERTa run A (all train) 0.5 fixed                                      0.409          0.439         0.689          0.872         0.817            1.0  0.825          0.23   714.0  164.0   
DistilRoBERTa run B (85% train) 0.5 fixed                                      0.402          0.439         0.652          0.872         0.811            1.0  0.806          0.23   714.0  164.0   
                                val-tuned global                      

CONTRACT RANKING (contracts sorted by predicted risk)


scope                                               contracts with >= 2 gold High clauses                                                                                                              \
framing                                                 classifier only (sum P high-tier)                                                                                                               
metric                                                                      recall@top10% ceiling@top10% recall@top20% ceiling@top20% recall@top30% ceiling@top30% pr_auc pr_auc_random n_items n_pos   
model                           threshold                                                                                                                                                               
TF-IDF + LR (all train)         0.5 fixed                                            0.45           0.55           0.7            1.0           0.7            1.0  0.719         0.196   102.0  20.0   
TF-IDF + LR (85% train)         val-tuned global                                     0.45           0.55           0.7            1.0           0.7            1.0  0.739         0.196   102.0  20.0   
                                val-tuned per-label                                  0.45           0.55           0.7            1.0           0.7            1.0  0.739         0.196   102.0  20.0   
DistilRoBERTa run A (all train) 0.5 fixed                                            0.45           0.55           0.8            1.0           0.8            1.0  0.783         0.196   102.0  20.0   
DistilRoBERTa run B (85% train) 0.5 fixed                                            0.45           0.55           0.7            1.0           0.8            1.0  0.727         0.196   102.0  20.0   
                                val-tuned global                                     0.45           0.55           0.7            1.0           0.8            1.0  0.727         0.196   102.0  20.0   
                                val-tuned per-label                                  0.45           0.55           0.7            1.0           0.8            1.0  0.727         0.196   102.0  20.0   

scope                                                                                                                                                                                          \
framing                                             with risk layer (triage rank)                                                                                                               
metric                                                              recall@top10% ceiling@top10% recall@top20% ceiling@top20% recall@top30% ceiling@top30% pr_auc pr_auc_random n_items n_pos   
model                           threshold                                                                                                                                                       
TF-IDF + LR (all train)         0.5 fixed                                    0.45           0.55          0.55            1.0          0.65            1.0  0.659         0.196   102.0  20.0   
TF-IDF + LR (85% train)         val-tuned global                             0.40           0.55          0.70            1.0          0.75            1.0  0.721         0.196   102.0  20.0   
                                val-tuned per-label                          0.40           0.55          0.60            1.0          0.70            1.0  0.652         0.196   102.0  20.0   
DistilRoBERTa run A (all train) 0.5 fixed                                    0.50           0.55          0.80            1.0          0.85            1.0  0.770         0.196   102.0  20.0   
DistilRoBERTa run B (85% train) 0.5 fixed                                    0.45           0.55          0.70            1.0          0.80            1.0  0.720         0.196   102.0  20.0   
                                val-tuned global   

## 10. Confidence intervals (contract bootstrap) for the headline classifier numbers

In [19]:
def f1_pair(T, B, K):
    T, B, K = T.astype(bool), B.astype(bool), K.astype(bool)
    tp = (T & B & K).sum(0); fp = (~T & B & K).sum(0); fn = (T & ~B & K).sum(0)
    micro = 2 * tp.sum() / (2 * tp.sum() + fp.sum() + fn.sum())
    sup = (tp + fn) > 0
    denom = 2 * tp + fp + fn
    f1 = np.divide(2 * tp, denom, out=np.zeros(len(tp), float), where=denom > 0)
    return micro, f1[sup].mean()


def bootstrap(P, thresholds, n=N_BOOTSTRAP, seed=0):
    rng = np.random.default_rng(seed)
    B = binarize(P, thresholds)
    draws = np.array([f1_pair(TEST_T[i], B[i], TEST_K[i]) for i in
                      (rng.integers(0, len(TEST_IDS), len(TEST_IDS)) for _ in range(n))])
    point = f1_pair(TEST_T, B, TEST_K)
    lo, hi = np.percentile(draws, [2.5, 97.5], axis=0)
    return {"micro_f1": point[0], "micro_f1_95ci": f"[{lo[0]:.3f}, {hi[0]:.3f}]",
            "macro_f1": point[1], "macro_f1_95ci": f"[{lo[1]:.3f}, {hi[1]:.3f}]"}

ci = pd.DataFrame({f"{m} | {t}": bootstrap(P, thr) for m, t, _, P, thr in configs}).T
display(ci)
ci.to_csv(OUT / "table_bootstrap_ci.csv")

,micro_f1,micro_f1_95ci,macro_f1,macro_f1_95ci
TF-IDF + LR (all train) | 0.5 fixed,0.682464,"[0.647, 0.716]",0.495121,"[0.443, 0.535]"
TF-IDF + LR (85% train) | val-tuned global,0.833603,"[0.815, 0.852]",0.72896,"[0.683, 0.758]"
TF-IDF + LR (85% train) | val-tuned per-label,0.827341,"[0.811, 0.844]",0.726837,"[0.687, 0.756]"
DistilRoBERTa run A (all train) | 0.5 fixed,0.84958,"[0.836, 0.864]",0.716935,"[0.674, 0.745]"
DistilRoBERTa run B (85% train) | 0.5 fixed,0.831146,"[0.813, 0.849]",0.6566,"[0.619, 0.683]"
DistilRoBERTa run B (85% train) | val-tuned global,0.816654,"[0.801, 0.833]",0.708243,"[0.677, 0.733]"
DistilRoBERTa run B (85% train) | val-tuned per-label,0.830648,"[0.813, 0.849]",0.735647,"[0.700, 0.760]"


## 11. Chunking, imbalance, thresholds: what was done and its measured effect

In [20]:
test_lab = chunk_labels[chunk_labels["split"] == "test"]
first = test_lab[test_lab["chunk_index"] == 0].set_index("contract_id")[CATEGORIES].fillna(False).astype(bool).reindex(TEST_IDS)
anyp = test_lab.groupby("contract_id")[CATEGORIES].max().fillna(False).astype(bool).reindex(TEST_IDS)
beyond = (anyp & ~first).to_numpy().sum() / anyp.to_numpy().sum()
cpc = chunks[chunks["split"] == "test"].groupby("contract_id").size()
print(f"Test contracts average {cpc.mean():.1f} chunks (median {cpc.median():.0f}, max {cpc.max()}).")
print(f"{beyond:.1%} of gold (contract, category) positives appear only after the first 510-token window.")

effects = results[(results["section"] == "classifier") & (results["scope"] == "34 modeled labels")
                  & results["metric"].isin(["micro_f1", "macro_f1"])].pivot_table(
    index=["model", "threshold"], columns="metric", values="value", sort=False).round(3)
display(effects)
per_category_tables[("DistilRoBERTa run A (all train)", "0.5 fixed")].to_csv(OUT / "per_category_runA.csv", index=False)
if RUN_VAL_MODEL:
    pd.DataFrame({"category": CATEGORIES, "threshold": B_per_label}).to_csv(OUT / "runB_val_thresholds.csv", index=False)
    per_category_tables[("DistilRoBERTa run B (85% train)", "val-tuned per-label")].to_csv(OUT / "per_category_runB_tuned.csv", index=False)

Test contracts average 28.8 chunks (median 16, max 170).
72.1% of gold (contract, category) positives appear only after the first 510-token window.


metric                                                                  micro_f1  macro_f1
model                                              threshold                              
TF-IDF + LR (all train)                            0.5 fixed               0.682     0.495
TF-IDF + LR (85% train)                            val-tuned global        0.834     0.729
                                                   val-tuned per-label     0.827     0.727
DistilRoBERTa run A (all train)                    0.5 fixed               0.850     0.717
DistilRoBERTa run B (85% train)                    0.5 fixed               0.831     0.657
                                                   val-tuned global        0.817     0.708
                                                   val-tuned per-label     0.831     0.736
DistilRoBERTa run A, first window only (chunkin... 0.5 fixed               0.429     0.138

## 12. Leakage and caveats to carry into any write-up

* Classifier thresholds: tuned only on validation contracts carved from the official train split (Run B). Test is scored once per configuration.
* The original High cutoff (0.70) in `configs/risk_rules.yaml` was picked by looking at test contracts. This notebook re-picks it on validation contracts (section 7) against a review share fixed in advance (`TARGET_REVIEW_SHARE`), so the risk-layer numbers here are held-out. The 20% target itself echoes the share the team saw on test; state it as a review-budget choice. Tiers and keywords were drawn from train-split samples (`Risk_clause_samples.ipynb`).
* The clause-ranking metric against T1 (high-tier category present) never depended on the cutoff, so it is clean either way.
* T2 ground truth is the rule engine applied to human labels, so risk-layer metrics measure agreement with the rules, not independent legal risk.
* Predicted evidence is a whole ~510-token chunk while gold evidence is the annotated span, so keyword hits can come from neighbouring clauses.
* `weak_detection` in the risk config was written from first-run test F1. Fine as documentation; do not use it to select models.
* 34 of 41 CUAD labels are modeled; 7 are below the 10% train cutoff and never predicted. Price Restrictions has 0 test positives.
* Run B's `run_metadata.json` still says 408 train contracts (hard-coded in the trainer); it actually trained on the reduced set.

In [21]:
print(json.dumps(leakage, indent=2))
# Print every results table into the notebook output so the saved .ipynb carries the results,
# whether this ran in the browser or through the VS Code Colab extension.
for path in sorted(OUT.glob("*.csv")):
    print(f"\n===== {path.name} =====")
    print(path.read_text())
import shutil
archive = shutil.make_archive("/content/evaluation_results", "zip", OUT)
try:
    from google.colab import files
    files.download(archive)
except Exception as exc:
    print(f"(browser download unavailable: {exc}); results are printed above and saved in {archive}")

{
  "contracts in both splits (by title)": 0,
  "test contracts whose full text also appears in train": 0,
  "test chunks whose exact text also appears in train": "0 / 2938"
}

===== all_metrics_long.csv =====
model,threshold,section,scope,framing,metric,value
TF-IDF + LR (all train),0.5 fixed,classifier,34 modeled labels,classifier,micro_precision,0.9515418502202643
TF-IDF + LR (all train),0.5 fixed,classifier,34 modeled labels,classifier,micro_recall,0.5320197044334976
TF-IDF + LR (all train),0.5 fixed,classifier,34 modeled labels,classifier,micro_f1,0.6824644549763033
TF-IDF + LR (all train),0.5 fixed,classifier,34 modeled labels,classifier,macro_precision,0.8297483009967366
TF-IDF + LR (all train),0.5 fixed,classifier,34 modeled labels,classifier,macro_recall,0.38860396983863454
TF-IDF + LR (all train),0.5 fixed,classifier,34 modeled labels,classifier,macro_f1,0.4951207808954398
TF-IDF + LR (all train),0.5 fixed,classifier,7 high-risk categories,classifier,micro_precision,0.9210526

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>